In [1]:
!rm -rf /content/ML-BIO-TECH

In [2]:
# REPOSITORY SETUP
# Clones the public ML-BIO-TECH repo (code + data) into the Colab
# session and adds model/ to the Python path so dataset_builder and
# train_model (written by earlier notebooks in this session) can be
# imported here.

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/adrianyadila/ML-BIO-TECH.git"
REPO_DIR = "/content/ML-BIO-TECH"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

PROJECT_ROOT = Path(REPO_DIR)
MODEL_DIR = PROJECT_ROOT / "model"
MODEL_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(MODEL_DIR))

print("MODEL_DIR added to PYTHONPATH")


MODEL_DIR added to PYTHONPATH


In [3]:
!pip install reportlab

In [4]:
"""
Ablation study runner

This script is responsible for:
1. Running feature-ablation experiments (EC only / KO only /
   KEGG pathways only / EC+KO / EC+KO+KEGG) across 4 classifiers
2. Caching each built dataset to avoid rebuilding on every run
3. Saving per-model result tables (CSV)
4. Generating bar-plot figures per model
5. Assembling a Nature-style PDF report summarizing all results

IMPORTANT:
- This script only evaluates models already defined in train_model.py
- No new model architectures are defined here
- No data leakage is introduced (scaling/CV happens inside train_model)

Author: Adriany Adila
"""

import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.pagesizes import A4
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.lib import colors

from dataset_builder import (
    build_ec_only_dataset,
    build_ko_only_dataset,
    build_kegg_only_dataset,
    build_ec_ko_dataset,
    build_final_dataset,
)

from train_model import (
    train_classifier_rf,
    train_classifier_nn,
    train_classifier_svm,
    train_classifier_xgb,
)

# PATHS & PARAMS

DATA_ROOT = os.path.join(PROJECT_ROOT, "data", "parquet_output_samples")

EC_ROOT = os.path.join(DATA_ROOT, "Annotation_Enzyme_Commission")
KO_ROOT = os.path.join(DATA_ROOT, "Annotation_KEGG_Orthology")
GCMS_ROOT = os.path.join(DATA_ROOT, "GC_MS_Metabolomics_Results")

KO_PATHWAY_MAPPING = os.path.join(PROJECT_ROOT, "model", "ko_to_kegg_pathway.tsv")

OUTDIR = os.path.join(PROJECT_ROOT, "model", "ablation_outputs")
os.makedirs(OUTDIR, exist_ok=True)

FIGDIR = os.path.join(OUTDIR, "figs")
os.makedirs(FIGDIR, exist_ok=True)

# DATASET CACHE
CACHE_DIR = os.path.join(OUTDIR, "cached_datasets")
os.makedirs(CACHE_DIR, exist_ok=True)

# SMART DATASET LOADING

def load_or_build_dataset(
    name,
    build_fn,
    rebuild=False,
):
    """
    Loads a cached dataset or builds it from scratch.

    Args:
        name: Dataset name
        build_fn: Function that builds the dataset
        rebuild: Forces a rebuild of this specific dataset
    """

    cache_file = os.path.join(
        CACHE_DIR,
        f"{name.replace(' ', '_')}.parquet"
    )

    if (not rebuild) and os.path.exists(cache_file):
        print(f"Loading cached dataset: {name}")
        return pd.read_parquet(cache_file)

    print(f"Building dataset from scratch: {name}")
    df = build_fn()

    if df.empty:
        print(f"  [WARN] Dataset {name} is empty!")
        return df

    print(f"  [OK] Dataset shape: {df.shape}")
    print(f"  [OK] Columns: {df.columns.tolist()}")

    df.to_parquet(cache_file, index=False)
    print("  [OK] Dataset cache updated.")

    return df

# STEP 2 — Ablation configurations

ABLATIONS = {
    "EC only": lambda: build_ec_only_dataset(
        ec_root=EC_ROOT,
        gcms_root=GCMS_ROOT,
    ),
    "KO only": lambda: build_ko_only_dataset(
        ko_root=KO_ROOT,
        gcms_root=GCMS_ROOT,
    ),
    "KEGG pathways only": lambda: build_kegg_only_dataset(
        ko_root=KO_ROOT,
        gcms_root=GCMS_ROOT,
        ko_pathway_mapping=KO_PATHWAY_MAPPING,
    ),
    "EC + KO": lambda: build_ec_ko_dataset(
        ec_root=EC_ROOT,
        ko_root=KO_ROOT,
        gcms_root=GCMS_ROOT,
    ),
    "EC + KO + KEGG": lambda: build_final_dataset(
        ec_root=EC_ROOT,
        ko_root=KO_ROOT,
        gcms_root=GCMS_ROOT,
        ko_pathway_mapping=KO_PATHWAY_MAPPING,
    ),
}

# STEP 3 — Dataset diagnostics

def dataset_diagnostics(df):
    print("\nDataset diagnostics")
    print(f"Samples : {len(df)}")
    print(f"Features: {df.shape[1]-2}")

    print("\nClass distribution")
    print(df["MES_label"].value_counts())

    print("\nClass proportions")
    print(df["MES_label"].value_counts(normalize=True).round(3))

# STEP 4 — Execute one ablation

def run_ablation(force_rebuild=False):
    """
    Runs the full ablation study.

    Args:
        force_rebuild: If True, forces a rebuild of ALL datasets
    """

    print("ABLATION STUDY")
    print(f"Force rebuild: {force_rebuild}")

    rf_rows = []
    nn_rows = []
    svm_rows = []
    xgb_rows = []

    total_ablation = len(ABLATIONS)
    current_ablation = 0

    for name, build_fn in ABLATIONS.items():
        current_ablation += 1
        print(f"\n[{current_ablation}/{total_ablation}] Running ablation: {name}")

        # Load dataset
        df = load_or_build_dataset(
            name,
            build_fn,
            rebuild=force_rebuild
        )

        # Skip empty datasets
        if df.empty:
            print(f"  [WARN] Dataset {name} is empty. Skipping...")
            continue

        # Check both classes are present
        if df["MES_label"].nunique() < 2:
            print(f"  [WARN] Dataset {name} has only one class. Skipping...")
            continue

        dataset_diagnostics(df)

        # RANDOM FOREST
        print("\n  Training Random Forest...")
        try:
            rf = train_classifier_rf(df)
            rf_rows.append({
                "Features": name,
                "Accuracy": rf["metrics"]["accuracy"],
                "Precision": rf["metrics"]["precision"],
                "Recall": rf["metrics"]["recall"],
                "F1": rf["metrics"]["f1"],
                "ROC_AUC": rf["metrics"]["roc_auc"]
            })
        except Exception as e:
            print(f"  [ERROR] Random Forest failed: {e}")

        # NEURAL NETWORK
        print("  Training Neural Network...")
        try:
            nn = train_classifier_nn(df)
            nn_rows.append({
                "Features": name,
                "Accuracy": nn["metrics"]["accuracy"],
                "Precision": nn["metrics"]["precision"],
                "Recall": nn["metrics"]["recall"],
                "F1": nn["metrics"]["f1"],
                "ROC_AUC": nn["metrics"]["roc_auc"]
            })
        except Exception as e:
            print(f"  [ERROR] Neural Network failed: {e}")

        # SVM
        print("  Training SVM...")
        try:
            svm = train_classifier_svm(df)
            svm_rows.append({
                "Features": name,
                "Accuracy": svm["metrics"]["accuracy"],
                "Precision": svm["metrics"]["precision"],
                "Recall": svm["metrics"]["recall"],
                "F1": svm["metrics"]["f1"],
                "ROC_AUC": svm["metrics"]["roc_auc"]
            })
        except Exception as e:
            print(f"  [ERROR] SVM failed: {e}")

        # XGBOOST
        print("  Training XGBoost...")
        try:
            xgb = train_classifier_xgb(df)
            xgb_rows.append({
                "Features": name,
                "Accuracy": xgb["metrics"]["accuracy"],
                "Precision": xgb["metrics"]["precision"],
                "Recall": xgb["metrics"]["recall"],
                "F1": xgb["metrics"]["f1"],
                "ROC_AUC": xgb["metrics"]["roc_auc"]
            })
        except Exception as e:
            print(f"  [ERROR] XGBoost failed: {e}")

    # RESULTS DATAFRAMES
    rf_df = pd.DataFrame(rf_rows).set_index("Features") if rf_rows else pd.DataFrame()
    nn_df = pd.DataFrame(nn_rows).set_index("Features") if nn_rows else pd.DataFrame()
    svm_df = pd.DataFrame(svm_rows).set_index("Features") if svm_rows else pd.DataFrame()
    xgb_df = pd.DataFrame(xgb_rows).set_index("Features") if xgb_rows else pd.DataFrame()

    # SAVE CSV
    if not rf_df.empty:
        rf_df.to_csv(os.path.join(OUTDIR, "rf.csv"))
        print(f"\n  [OK] RF results saved: {len(rf_df)} configurations")
    else:
        print("\n  [WARN] No RF results to save")

    if not nn_df.empty:
        nn_df.to_csv(os.path.join(OUTDIR, "nn.csv"))
        print(f"  [OK] NN results saved: {len(nn_df)} configurations")

    if not svm_df.empty:
        svm_df.to_csv(os.path.join(OUTDIR, "svm.csv"))
        print(f"  [OK] SVM results saved: {len(svm_df)} configurations")

    if not xgb_df.empty:
        xgb_df.to_csv(os.path.join(OUTDIR, "xgb.csv"))
        print(f"  [OK] XGB results saved: {len(xgb_df)} configurations")

    # PRINT SUMMARY
    print("\n")
    print("SUMMARY - MEAN F1-SCORE")

    if not rf_df.empty and "F1" in rf_df.columns:
        print(f"Random Forest : F1={rf_df['F1'].mean():.3f} | AUC={rf_df['ROC_AUC'].mean():.3f} | n={len(rf_df)}")
    else:
        print("Random Forest : NO RESULTS")

    if not nn_df.empty and "F1" in nn_df.columns:
        print(f"Neural Network: F1={nn_df['F1'].mean():.3f} | AUC={nn_df['ROC_AUC'].mean():.3f} | n={len(nn_df)}")
    else:
        print("Neural Network: NO RESULTS")

    if not svm_df.empty and "F1" in svm_df.columns:
        print(f"SVM           : F1={svm_df['F1'].mean():.3f} | AUC={svm_df['ROC_AUC'].mean():.3f} | n={len(svm_df)}")
    else:
        print("SVM           : NO RESULTS")

    if not xgb_df.empty and "F1" in xgb_df.columns:
        print(f"XGBoost       : F1={xgb_df['F1'].mean():.3f} | AUC={xgb_df['ROC_AUC'].mean():.3f} | n={len(xgb_df)}")
    else:
        print("XGBoost       : NO RESULTS")

    return rf_df, nn_df, svm_df, xgb_df

# FIGURES

def barplot(df, metric, fname):
    """Generates a bar plot with safety checks."""
    if df.empty:
        print(f"  [WARN] Cannot plot {fname}: DataFrame is empty")
        return

    if metric not in df.columns:
        print(f"  [WARN] Cannot plot {fname}: column '{metric}' not found")
        print(f"  Available columns: {df.columns.tolist()}")
        return

    plt.figure(figsize=(6,4))
    df[metric].plot(kind="bar")
    plt.ylabel(metric)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGDIR, fname), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  [OK] Saved figure: {fname}")

# PDF GENERATION

def format_df_2decimals(df):
    """Formats a DataFrame with 2 decimal places."""
    if df.empty:
        return df

    df_fmt = df.copy()
    for col in df_fmt.columns:
        if pd.api.types.is_numeric_dtype(df_fmt[col]):
            df_fmt[col] = df_fmt[col].map(
                lambda x: f"{x:.2f}" if pd.notnull(x) else "NaN"
            )
    return df_fmt

def df_table(df):
    """Converts a DataFrame to a ReportLab table."""
    if df.empty:
        # Returns a table with a "No data" message
        data = [["No results available"]]
        t = Table(data, hAlign="LEFT")
        t.setStyle(TableStyle([
            ("FONT", (0,0), (-1,-1), "Helvetica"),
            ("FONTSIZE", (0,0), (-1,-1), 10),
            ("ALIGN", (0,0), (-1,-1), "CENTER"),
            ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
        ]))
        return t

    df_fmt = format_df_2decimals(df)
    df_fmt = df_fmt.reset_index()

    data = [df_fmt.columns.tolist()] + df_fmt.values.tolist()

    t = Table(data, hAlign="LEFT", repeatRows=1)

    t.setStyle(TableStyle([
        # Header
        ("BACKGROUND", (0,0), (-1,0), colors.HexColor("#F2F2F2")),
        ("FONT", (0,0), (-1,0), "Helvetica-Bold"),
        ("TEXTCOLOR", (0,0), (-1,0), colors.black),
        ("LINEBELOW", (0,0), (-1,0), 1, colors.black),
        # Body
        ("FONT", (0,1), (-1,-1), "Helvetica"),
        ("FONTSIZE", (0,0), (-1,-1), 9),
        ("ALIGN", (1,1), (-1,-1), "CENTER"),
        ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
        ("LINEBELOW", (0,1), (-1,-1), 0.25, colors.lightgrey),
        ("TOPPADDING", (0,0), (-1,-1), 6),
        ("BOTTOMPADDING", (0,0), (-1,-1), 6),
        ("LEFTPADDING", (0,0), (-1,-1), 6),
        ("RIGHTPADDING", (0,0), (-1,-1), 6),
    ]))

    return t

def make_pdf(rf_df, nn_df, svm_df, xgb_df):
    """Generates the PDF with safety checks."""

    styles = getSampleStyleSheet()

    styles.add(ParagraphStyle(
        name="NatureTitle",
        parent=styles["Title"],
        fontSize=20,
        leading=24,
        alignment=TA_CENTER,
        spaceAfter=14
    ))

    styles.add(ParagraphStyle(
        name="NatureSubtitle",
        parent=styles["Normal"],
        fontSize=11,
        leading=15,
        alignment=TA_CENTER,
        textColor=colors.grey,
        spaceAfter=22
    ))

    styles.add(ParagraphStyle(
        name="NatureSection",
        parent=styles["Heading2"],
        fontSize=12,
        leading=15,
        spaceBefore=18,
        spaceAfter=6
    ))

    styles.add(ParagraphStyle(
        name="NatureBody",
        parent=styles["Normal"],
        fontSize=10,
        leading=14,
        alignment=TA_LEFT,
        spaceAfter=12
    ))

    styles.add(ParagraphStyle(
        name="NatureCaption",
        parent=styles["Normal"],
        fontSize=9,
        leading=12,
        textColor=colors.grey,
        spaceBefore=6,
        spaceAfter=14
    ))

    doc = SimpleDocTemplate(
        os.path.join(OUTDIR, "Ablation_Study_Nature.pdf"),
        pagesize=A4,
        rightMargin=42,
        leftMargin=42,
        topMargin=40,
        bottomMargin=40
    )

    story = []

    # TITLE
    story.append(Paragraph(
        "Ablation Study of Multi-Omics Features for Functional Aromatic Metabolism Classification",
        styles["NatureTitle"]
    ))

    story.append(Paragraph(
        "Systematic evaluation of EC, KO and KEGG pathway features for binary classification of functional aromatic metabolism.",
        styles["NatureSubtitle"]
    ))

    story.append(Paragraph(
        "We performed a systematic ablation study to quantify the contribution "
        "of Enzyme Commission (EC), KEGG Orthology (KO), and pathway-level "
        "features for predicting functional aromatic metabolism. "
        "The prediction task was formulated as a binary classification problem, "
        "where biosamples were classified according to the presence or absence "
        "of strong evidence of aromatic metabolism based on GC-MS-derived "
        "Metabolic Evidence Scores (MES). "
        "Model performance was evaluated using stratified 5-fold cross-validation "
        "and reported using Accuracy, Precision, Recall, F1-score and ROC-AUC.",
        styles["NatureBody"]
    ))

    # RANDOM FOREST
    story.append(Paragraph("Random Forest Classification", styles["NatureSection"]))
    story.append(df_table(rf_df))
    story.append(Paragraph(
        "Table 1 | Random Forest classification performance across feature ablations.",
        styles["NatureCaption"]
    ))
    story.append(Spacer(1,16))

    # NEURAL NETWORK
    story.append(Paragraph("Neural Network Classification", styles["NatureSection"]))
    story.append(df_table(nn_df))
    story.append(Paragraph(
        "Table 2 | Neural Network classification performance across feature ablations.",
        styles["NatureCaption"]
    ))
    story.append(PageBreak())

    # SVM
    story.append(Paragraph("Support Vector Machine Classification", styles["NatureSection"]))
    story.append(df_table(svm_df))
    story.append(Paragraph(
        "Table 3 | Support Vector Machine classification performance across feature ablations.",
        styles["NatureCaption"]
    ))
    story.append(Spacer(1,16))

    # XGBOOST
    story.append(Paragraph("XGBoost Classification", styles["NatureSection"]))
    story.append(df_table(xgb_df))
    story.append(Paragraph(
        "Table 4 | XGBoost classification performance across feature ablations.",
        styles["NatureCaption"]
    ))

    doc.build(story)

# MAIN

if __name__ == "__main__":

    # CONFIGURATION: force_rebuild
    # force_rebuild=False: USE CACHE (fast)
    # force_rebuild=True:  FULL REBUILD (slow)

    FORCE_REBUILD = True

    if FORCE_REBUILD:
        print("\n  FORCE_REBUILD = True")
        print("   Rebuilding ALL datasets from scratch...")
    else:
        print("\n FORCE_REBUILD = False")
        print("   Using cached datasets (fast).")

    print("\n" + "=" * 70)

    # Run the ablation
    rf_df, nn_df, svm_df, xgb_df = run_ablation(force_rebuild=FORCE_REBUILD)

    # Generate figures
    print("\n" + "=" * 70)
    print("GENERATING FIGURES")
    print("=" * 70)

    barplot(rf_df, "F1", "rf_f1.png")
    barplot(nn_df, "F1", "nn_f1.png")
    barplot(svm_df, "F1", "svm_f1.png")
    barplot(xgb_df, "F1", "xgb_f1.png")

    # Generate PDF
    print("\n" + "=" * 70)
    print("GENERATING PDF")
    print("=" * 70)

    make_pdf(rf_df, nn_df, svm_df, xgb_df)
    print("\n PDF generated: Ablation_Study_Nature.pdf")



  FORCE_REBUILD = True
   Rebuilding ALL datasets from scratch...

ABLATION STUDY
Force rebuild: True

[1/5] Running ablation: EC only
Building dataset from scratch: EC only

MES distribution (this study)
count    437.000000
mean       0.071250
std        0.057179
min        0.000000
25%        0.030822
50%        0.048755
75%        0.096685
max        0.321944
Name: MES, dtype: float64

Per-study quantile used : 0.5
Per-study threshold     : 0.048755

MES_label distribution (this study)
MES_label
1    219
0    218
Name: count, dtype: int64
Proportion positive     : 0.501
Biosamples with MES == 0: 3

MES distribution (this study)
count    22.000000
mean      0.011727
std       0.003344
min       0.006918
25%       0.009413
50%       0.011303
75%       0.013158
max       0.020011
Name: MES, dtype: float64

Per-study quantile used : 0.5
Per-study threshold     : 0.011303

MES_label distribution (this study)
MES_label
0    11
1    11
Name: count, dtype: int64
Proportion positive     : 0

In [5]:
"""
Leave-One-Study-Out (LOSO) cross-validation
--------------------------------------------

Purpose:
    Test whether the functional features (EC / KO / pathway) carry a
    signal about aromatic-degradation activity (MES_label) that
    GENERALIZES across studies, as opposed to a signal that merely
    separates the three studies (batch effect).

Design:
    For each study S in {study_1, study_2, study_3}:
        - train on the OTHER two studies
        - test on S (never seen during training)
    Scaling is fit on the training studies only (no leakage).

Interpretation:
    - Test ROC-AUC clearly > 0.5 on held-out studies
        -> signal generalizes across batches -> real degradation signal
    - Test ROC-AUC ~ 0.5 (collapses)
        -> features do not predict degradation once study is held out
        -> apparent performance in random CV was batch confounding

Because MES_label is defined per study (median split within each study),
every held-out study is ~50% positive, so the chance baseline is
ROC-AUC = 0.5 and accuracy ~ 0.5.
"""

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

RANDOM_STATE = 42


def _build_models():
    """Same model configs as the main pipeline (minus the MLP, which
    is unusable at this n). Each is a scaler+model pipeline so scaling
    is fit on train studies only."""
    return {
        "RANDOM FOREST": Pipeline([
            ("scaler", StandardScaler()),
            ("model", RandomForestClassifier(
                n_estimators=200,
                max_depth=6,
                min_samples_leaf=3,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            )),
        ]),
        "SVM": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVC(
                kernel="rbf",
                C=1.0,
                gamma="scale",
                probability=True,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            )),
        ]),
        "XGBOOST": Pipeline([
            ("scaler", StandardScaler()),
            ("model", XGBClassifier(
                n_estimators=100,
                max_depth=3,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                verbosity=0,
            )),
        ]),
    }


def split_features_targets(df: pd.DataFrame):
    """Same convention as train_model.py."""
    drop_cols = ["biosample", "study", "MES", "MES_label"]
    feature_cols = [c for c in df.columns if c not in drop_cols]
    X = df[feature_cols]
    y = df["MES_label"]
    return X, y, feature_cols


def run_loso(df: pd.DataFrame, feature_set_name: str = ""):
    """
    Runs leave-one-study-out CV for all models on a given dataset.

    df must contain: 'study', 'MES_label', and feature columns.
    Returns a tidy DataFrame of per-fold and averaged results.
    """
    studies = sorted(df["study"].unique())
    if len(studies) < 2:
        raise ValueError("LOSO needs at least 2 studies.")

    X_all, y_all, feature_cols = split_features_targets(df)

    models = _build_models()
    rows = []

    print("=" * 64)
    print(f"LOSO — {feature_set_name}")
    print(f"Studies: {studies}")
    print(f"Features ({len(feature_cols)}): {feature_cols}")
    print("=" * 64)

    for model_name, pipe in models.items():
        print(f"\n{'-' * 64}\n{model_name}\n{'-' * 64}")

        fold_aucs = []
        fold_accs = []
        fold_f1s = []

        for test_study in studies:
            train_mask = df["study"] != test_study
            test_mask = df["study"] == test_study

            X_tr, y_tr = X_all[train_mask], y_all[train_mask]
            X_te, y_te = X_all[test_mask], y_all[test_mask]

            # Guard: need both classes in train and test to score AUC
            if y_tr.nunique() < 2 or y_te.nunique() < 2:
                print(f"  test={test_study}: skipped "
                      f"(train classes={y_tr.nunique()}, "
                      f"test classes={y_te.nunique()})")
                continue

            pipe.fit(X_tr, y_tr)

            y_pred = pipe.predict(X_te)
            try:
                y_prob = pipe.predict_proba(X_te)[:, 1]
                auc = roc_auc_score(y_te, y_prob)
            except Exception:
                auc = np.nan

            acc = accuracy_score(y_te, y_pred)
            f1 = f1_score(y_te, y_pred, zero_division=0)
            prec = precision_score(y_te, y_pred, zero_division=0)
            rec = recall_score(y_te, y_pred, zero_division=0)

            fold_aucs.append(auc)
            fold_accs.append(acc)
            fold_f1s.append(f1)

            print(f"  test={test_study:<18} "
                  f"n_test={test_mask.sum():>3}  "
                  f"AUC={auc:.3f}  acc={acc:.3f}  "
                  f"f1={f1:.3f}  prec={prec:.3f}  rec={rec:.3f}")

            rows.append({
                "feature_set": feature_set_name,
                "model": model_name,
                "test_study": test_study,
                "n_test": int(test_mask.sum()),
                "roc_auc": auc,
                "accuracy": acc,
                "f1": f1,
                "precision": prec,
                "recall": rec,
            })

        if fold_aucs:
            print(f"  {'MEAN':<18} "
                  f"        AUC={np.nanmean(fold_aucs):.3f}  "
                  f"acc={np.mean(fold_accs):.3f}  "
                  f"f1={np.mean(fold_f1s):.3f}")

    results = pd.DataFrame(rows)

    # Averaged summary per model
    if not results.empty:
        print("\n" + "=" * 64)
        print(f"SUMMARY (mean across held-out studies) — {feature_set_name}")
        print("=" * 64)
        summary = (
            results
            .groupby("model")[["roc_auc", "accuracy", "f1"]]
            .mean()
            .round(3)
        )
        print(summary)
        print("\nChance baseline: ROC-AUC = 0.5, accuracy ~ 0.5")

    return results


# Build one dataset per feature-ablation set, then run LOSO on each
# so results are comparable across feature sets.

df_ec = build_ec_only_dataset(
    ec_root=EC_ROOT,
    gcms_root=GCMS_ROOT,
)

df_ko = build_ko_only_dataset(
    ko_root=KO_ROOT,
    gcms_root=GCMS_ROOT,
)

df_kegg = build_kegg_only_dataset(
    ko_root=KO_ROOT,
    gcms_root=GCMS_ROOT,
    ko_pathway_mapping=KO_PATHWAY_MAPPING,
)

df_full = build_final_dataset(
    ec_root=EC_ROOT,
    ko_root=KO_ROOT,
    gcms_root=GCMS_ROOT,
    ko_pathway_mapping=KO_PATHWAY_MAPPING,
)

all_results = pd.concat([
    run_loso(df_ec, "EC only"),
    run_loso(df_ko, "KO only"),
    run_loso(df_kegg, "KEGG only"),
    run_loso(df_full, "EC + KO + KEGG"),
], ignore_index=True)



MES distribution (this study)
count    437.000000
mean       0.071250
std        0.057179
min        0.000000
25%        0.030822
50%        0.048755
75%        0.096685
max        0.321944
Name: MES, dtype: float64

Per-study quantile used : 0.5
Per-study threshold     : 0.048755

MES_label distribution (this study)
MES_label
1    219
0    218
Name: count, dtype: int64
Proportion positive     : 0.501
Biosamples with MES == 0: 3

MES distribution (this study)
count    22.000000
mean      0.011727
std       0.003344
min       0.006918
25%       0.009413
50%       0.011303
75%       0.013158
max       0.020011
Name: MES, dtype: float64

Per-study quantile used : 0.5
Per-study threshold     : 0.011303

MES_label distribution (this study)
MES_label
0    11
1    11
Name: count, dtype: int64
Proportion positive     : 0.500
Biosamples with MES == 0: 0

MES distribution (this study)
count    34.000000
mean      0.003843
std       0.002189
min       0.001112
25%       0.002448
50%       0.0035